# ESML v2: one-line pipeline creation
Install the local `azure-esml-sdk` wheel or repository package first. `lake_settings.json` is customer-owned: replace workspace, compute, environment and datastore with existing resources. This notebook only renders; it never submits or provisions Azure resources.

The example maps two CSV source folders containing disjoint partitions of the Kaggle Pima dataset. Download/review the data separately with the model-factory ingestion tools. This historical diabetes dataset is educational, not clinically representative or suitable for diagnosis.

In [ ]:
from pathlib import Path
from uuid import uuid4
from azure_esml import ESMLProject, PipelineRequest, PipelineType
project = ESMLProject.from_json(Path('lake_settings.json'))
request = PipelineRequest(data_date_utc='2026-09-13', run_id='training-' + uuid4().hex)


In [ ]:
plan = project.create_pipeline(PipelineType.IN_2_GOLD_TRAINING_AUTOML, request, output=Path('generated') / request.run_id)
print(plan.yaml_path)
print(plan.document['experiment_name'])
list(plan.document['jobs'])


Change only the pipeline type to `IN_2_GOLD_TRAINING_MANUAL` for the custom estimator. Dataset folders determine input fan-out; changing folder count regenerates the graph. New settings default to raw IN-to-BRONZE and validated BRONZE-to-SILVER steps, followed by a separate use-case gold even for one dataset. Silver/gold default to Delta; explicitly select `table_format: parquet` for compatibility.

The returned YAML and `plan.to_sdk()` represent the same v2 pipeline. Submission is explicit: construct `AzureMLSDKBackend.from_cli(project.target)` or `AzureMLCLIBackend(project.target)`, inject it into a new `ESMLProject`, and call `execute_pipeline(plan)`. Cloud execution incurs charges. Environment dependencies and private lake access must already be provisioned.

In [ ]:
sdk_job = plan.to_sdk()
assert sdk_job.type == 'pipeline'
print('Loaded the generated pipeline with Azure ML SDK v2; no job was submitted.')
